In [ ]:
!pip install unsloth datasets transformers trl peft accelerate bitsandbytes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import files
uploaded = files.upload()
# Upload your training_data.jsonl here

Saving training_data.jsonl to training_data.jsonl


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,   # saves GPU memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r                = 16,
    target_modules   = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    lora_alpha       = 16,
    lora_dropout     = 0,
    bias             = "none",
    use_gradient_checkpointing = "unsloth",
    random_state     = 42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.8 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = "training_data.jsonl",
    split      = "train"
)

# Train/eval split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset["train"]
eval_data  = dataset["test"]

print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

# The dataset already contains the formatted text in the 'text' field,
# so no further formatting is needed for SFTTrainer.
# The 'instruction' and 'response' keys are not present in the loaded data.
# Keeping the original format_prompt function for reference but it's not used.
def format_prompt(row):
    return {
        "text": f"""<|im_start|>system\nYou are a helpful assistant for an institutional profile system.\nAnswer questions accurately based on the member's data.<|im_end|>\n<|im_start|>user\n{row['instruction']}<|im_end|>\n<|im_start|>assistant\n{row['response']}<|im_end|>"""
    }


Generating train split: 0 examples [00:00, ? examples/s]

Train: 10310 | Eval: 1146


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = train_data,
    eval_dataset     = eval_data,
    dataset_text_field = "text",
    max_seq_length   = 2048,
    args = TrainingArguments(
        output_dir              = "./qwen_finetuned",
        num_train_epochs        = 3,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps            = 10,
        learning_rate           = 2e-4,
        fp16                    = not torch.cuda.is_bf16_supported(),
        bf16                    = torch.cuda.is_bf16_supported(),
        logging_steps           = 10,
        # evaluation_strategy     = "epoch", # Removed to fix TypeError
        # save_strategy           = "epoch", # Removed to fix TypeError
        # load_best_model_at_end  = True,    # Removed as it depends on evaluation_strategy
        optim                   = "adamw_8bit",
        weight_decay            = 0.01,
        lr_scheduler_type       = "linear",
        seed                    = 42,
        report_to               = "none",
    ),
)

trainer_stats = trainer.train()
print(f"Training loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10310 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1146 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,310 | Num Epochs = 3 | Total steps = 1,935
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.993077
20,1.178308
30,0.573952
40,0.382726
50,0.326815
60,0.293864
70,0.286381
80,0.241438
90,0.231759
100,0.224542


Unsloth: Restored added_tokens_decoder metadata in ./qwen_finetuned/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen_finetuned/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen_finetuned/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen_finetuned/checkpoint-1935/tokenizer_config.json.


Training loss: 0.1413


In [ ]:
# Explanation for NameError: 'model' is not defined.
# This error indicates that the 'model' and 'tokenizer' variables,
# which are defined in previous cells (e.g., RLm4XRft-Xbc and OsOZPS2vA5H8),
# were not available when this cell was executed. This can happen if the
# Python runtime was restarted or disconnected, or if those cells were not
# run successfully.
# Please ensure that cells from 'RLm4XRft-Xbc' to 'OsOZPS2vA5H8' are
# executed before running this cell.

# Save LoRA adapter first
model.save_pretrained("qwen_lora_adapter")
tokenizer.save_pretrained("qwen_lora_adapter")

# Export to GGUF for llama.cpp (what your local server uses)
model.save_pretrained_gguf(
    "qwen_finetuned_gguf",
    tokenizer,
    quantization_method = "q4_k_m"   # best quality/size tradeoff
)


Unsloth: Restored added_tokens_decoder metadata in qwen_lora_adapter/tokenizer_config.json.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen_finetuned_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:14<00:00, 14.49s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:06<00:00,  6.48s/it]


Unsloth: Merge process complete. Saved to `/content/qwen_finetuned_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen_finetuned_gguf_gguf/qwen2.5-0.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsl

{'save_directory': 'qwen_finetuned_gguf',
 'gguf_directory': 'qwen_finetuned_gguf_gguf',
 'gguf_files': ['qwen_finetuned_gguf_gguf/qwen2.5-0.5b-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'qwen_finetuned_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
# Download the GGUF file
from google.colab import files
import os

output_dir = "qwen_finetuned_gguf"

if not os.path.exists(output_dir):
    print(f"Error: Directory '{output_dir}' not found. Please ensure the model was successfully exported to GGUF in the previous step (cell a4W2Y-rvIDtz).")
else:
    gguf_files = [f for f in os.listdir(output_dir) if f.endswith(".gguf")]
    if not gguf_files:
        print(f"No .gguf files found in '{output_dir}'. Please ensure the model was successfully exported to GGUF in the previous step (cell a4W2Y-rvIDtz).")
    else:
        for f in gguf_files:
            file_path = os.path.join(output_dir, f)
            files.download(file_path)
            print(f"Downloaded: {f}")

No .gguf files found in 'qwen_finetuned_gguf'. Please ensure the model was successfully exported to GGUF in the previous step (cell a4W2Y-rvIDtz).
